In [ ]:
from transformers import AutoTokenizer, AutoModel, AutoModelForTokenClassification, TrainingArguments, Trainer, pipeline
import platform
import torch
import numpy as np
import pandas as pd

/opt/anaconda3/envs/NLP-bio/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
model_name = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"

label_list = []
label2id =  {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label_list),
    label2id = label2id,
    id2label = id2label,

)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps") #might have issues on macbook.
else:
    device = torch.device("cpu")
model.to(device)


Loading tokenizer...
Loading model to CPU...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 76476.68it/s]
BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Moving to MPS (Mac GPU)...
Success! Model is on Mac GPU.


In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy = "epoch",
    learning_rate = 2e-5,
    per_device_train_batch_size = 16,
    num_train_epochs = 3,
    weight_decay=0.01,
    use_mps_device=True if device.type == "mps" else False
)

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset, #need training set
    eval_dataset = dev_dataset,
    tokenizer = tokenizer
)

In [ ]:
NER_Pipeline = pipeline(
    "ner",
    model = model,
    tokenizer=tokenizer,
    device = device 
    )